# YAMNet 冻结版 + LightGBM 嵌入导出 (Kaggle 一键版)
作者: Wenjuan Huang

## 设计思路
- **冻结 YAMNet**: 不微调, 只前向, 一次性算完全部嵌入 (~30-60 min)
- **流式 tf.data**: 用 tf.py_function 包装 librosa, 内存恒定 ~MB 级
- **低资源物种增强**: 对 <15 样本的物种调用 expand_with_augmentation (time stretch / pitch shift / 加噪 / volume), **按折独立进行** (仅用本折训练集, 防止验证泄漏)
- **5 折 Dense 头训练**: 在 3072 维嵌入上训练轻量分类头, 综合评估 (Top-1/5, macro-F1, balanced acc, 类频率分层)
- **LightGBM 导出**: 每折存 .npz (含 embeddings + 字符串标签 + filename + 类权重 + 该折的增强样本)
- **噪声鲁棒性评估**: 流式叠噪 + 重算嵌入, 综合指标
- **推理速度测试**

## 数据集 (Kaggle input)
- processeddata-129834/ 上传为 Kaggle input, 含:
  - 02_train_full_weighted.csv (129,834 训练样本, 含 sampler_weight / loss_class_weight)
  - 03_test_holdout.csv (14,416 留出测试样本)
  - cv_fold{1-5}_train/val.csv (5 折 CV)
  - class_weights.csv (每类预计算权重)
  - 1,126 个鸟类
- BirdCLEF 2021-2026 年度音频数据集挂载为 input

## Cell 1/7: 依赖与配置

In [ ]:
# 依赖: Kaggle 默认已装 tensorflow, tensorflow_hub, librosa, pandas, numpy
get_ipython().system("pip install resampy -q  # librosa kaiser_fast needs resampy")
import re
import os
import json
import time
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import librosa
from sklearn.metrics import (accuracy_score, top_k_accuracy_score,
                             f1_score, balanced_accuracy_score)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


class Cfg:
    # YAMNet 输入: 16kHz 单声道 float32 [-1,1]
    SR = 16000
    CLIP_SECONDS = 5.0
    CLIP_SAMPLES = int(SR * CLIP_SECONDS)          # 80000

    # Kaggle 路径
    INPUT_BASE = "/kaggle/input"
    OUT_DIR = "/kaggle/working/yamnet"
    EMB_DIR = "/kaggle/working/yamnet_embeddings"  # 给 LightGBM 队友 (契约目录)
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(EMB_DIR, exist_ok=True)

    # processeddata-129834 中的 CSV 文件名
    TRAIN_WEIGHTED_CSV = "02_train_full_weighted.csv"
    TEST_CSV = "03_test_holdout.csv"
    CLASS_WEIGHTS_CSV = "class_weights.csv"
    N_FOLDS = 5

    @staticmethod
    def fold_csvs(fold):
        return (f"cv_fold{fold}_train.csv", f"cv_fold{fold}_val.csv")

    # 音频目录候选 (不同年份命名不同)
    AUDIO_ROOT_CANDIDATES = ("train_audio", "train_short_audio")

    # YAMNet 模型 (首次联网下载, ~17MB)
    YAMNET_HANDLE = "https://tfhub.dev/google/yamnet/1"

    # 嵌入聚合模式: "mean" (1024) / "mean_max" (2048) / "mean_max_std" (3072)
    # 契约推荐 mean_max_std -> 3072 维
    POOL_MODE = "mean_max_std"
    _POOL_FACTORS = {"mean": 1, "mean_max": 2, "mean_max_std": 3}
    EMBED_DIM = 1024 * _POOL_FACTORS[POOL_MODE]   # 3072

    # 嵌入提取 batch size (冻结前向, 可以开大)
    EMB_BATCH_SIZE = 128
    # Dense 头训练
    HEAD_BATCH_SIZE = 256
    HEAD_EPOCHS = 50
    HEAD_LR = 1e-3
    DROPOUT = 0.3
    SEED = 42

    # 低资源物种增强
    LOW_RESOURCE_THRESHOLD = 15       # < 此数量的训练样本视为低资源
    AUG_TARGET_PER_SPECIES = 15       # 增强后每物种目标样本数
    AUG_MAX_PER_SPECIES = 50          # 每物种最多新增样本数

    # 类别频率分组边界 (用于分层评估)
    FREQ_GROUP_BINS = [0, 15, 50, 200, float("inf")]
    FREQ_GROUP_NAMES = ["tail(<15)", "low(15-49)", "mid(50-199)", "head(>=200)"]

    # 缓存路径 (内部用)
    EMB_CACHE = os.path.join(OUT_DIR, "embeddings_all.npz")
    NOISE_EMB_CACHE = os.path.join(OUT_DIR, "noise_embeddings.npz")
    LABEL_MAP_PATH = os.path.join(OUT_DIR, "label_map.json")


cfg = Cfg()

## Cell 2/7: Kaggle 挂载扫描 + CSV 加载 + 标签映射

In [ ]:
def print_mounted_inputs(input_base=cfg.INPUT_BASE, max_per_dir=8):
    "列出 /kaggle/input 下挂载的数据集及其一层子目录。"
    path = Path(input_base)
    if not path.exists():
        print(f"[挂载] {path} 不存在")
        return
    entries = sorted(p for p in path.iterdir() if p.is_dir())
    if not entries:
        print(f"[挂载] {path} 下无子目录")
        return
    print(f"[挂载] {path} 下发现 {len(entries)} 个数据集:")
    for d in entries:
        sub = [c.name for c in d.iterdir() if c.is_dir()][:max_per_dir]
        print(f"  - {d.name}  子目录: {sub}")


def scan_kaggle_inputs(input_base=cfg.INPUT_BASE):
    "扫描 /kaggle/input 一次, 收集音频根目录与目标 CSV 路径。遇音频根剪枝。"
    target_csvs = {cfg.TRAIN_WEIGHTED_CSV, cfg.TEST_CSV, cfg.CLASS_WEIGHTS_CSV}
    for f in range(1, cfg.N_FOLDS + 1):
        target_csvs.update(cfg.fold_csvs(f))

    year2root = {}
    unknown_roots = []
    csv_paths = {}

    if not Path(input_base).exists():
        return year2root, unknown_roots, csv_paths

    for root, dirs, files in os.walk(input_base):
        base = os.path.basename(root)
        if base in cfg.AUDIO_ROOT_CANDIDATES:
            rp = Path(root)
            m = re.search(r"(20\d{2})", root)
            if m:
                year2root.setdefault(int(m.group(1)), rp)
            else:
                if rp not in unknown_roots:
                    unknown_roots.append(rp)
            dirs[:] = []
            continue
        for f in files:
            if f in target_csvs and f not in csv_paths:
                csv_paths[f] = Path(root) / f

    for r in unknown_roots:
        year2root.setdefault(-1, r)
    return year2root, unknown_roots, csv_paths


def parse_source_years(s):
    "解析 source_year 字段 (int 或 str), 返回降序年份列表。"
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return []
    years = re.findall(r"20\d{2}", str(s))
    return sorted({int(y) for y in years}, reverse=True)


def resolve_audio_path(row, year2root, all_roots):
    "依据 filename / primary_label / source_year 定位音频文件。"
    fname = str(row["filename"]).replace("\\", "/")
    primary_label = str(row["primary_label"])
    basename = os.path.basename(fname)

    roots_to_try = []
    for y in parse_source_years(row.get("source_year")):
        r = year2root.get(y)
        if r is not None and r not in roots_to_try:
            roots_to_try.append(r)
    for r in all_roots:
        if r not in roots_to_try:
            roots_to_try.append(r)

    for root in roots_to_try:
        for cand in (os.path.join(root, fname),
                     os.path.join(root, primary_label, basename),
                     os.path.join(root, basename)):
            if cand and os.path.exists(cand):
                return cand
    return None


def build_label_map(csv_paths):
    "构建 label2idx / idx2label (train + val_fold1 + test 并集)。"
    label_frames = []
    for fname in [cfg.TRAIN_WEIGHTED_CSV, cfg.TEST_CSV, cfg.fold_csvs(1)[1]]:
        if fname in csv_paths:
            label_frames.append(
                pd.read_csv(csv_paths[fname], usecols=["primary_label"])["primary_label"])
    classes = sorted(pd.concat(label_frames).astype(str).unique().tolist())
    label2idx = {c: i for i, c in enumerate(classes)}
    idx2label = {i: c for c, i in label2idx.items()}

    with open(cfg.LABEL_MAP_PATH, "w", encoding="utf-8") as f:
        json.dump({"label2idx": label2idx,
                   "idx2label": {str(k): v for k, v in idx2label.items()},
                   "classes": classes}, f, ensure_ascii=False, indent=2)
    print(f"[标签] {len(classes)} 类, 映射已存: {cfg.LABEL_MAP_PATH}")
    return classes, label2idx, idx2label

## Cell 3/7: 流式音频解码 + 冻结 YAMNet 嵌入提取

核心: 内存恒定 ~MB 级
- tf.py_function 包装 librosa.load, 在 CPU 上并行解码 OGG
- 冻结 YAMNet (trainable=False) 只做前向, batch=128 在 GPU 上很快
- 一次性算完全部 ~144K 唯一文件, 缓存到 .npz (~590 MB)
- compute_embeddings_from_arrays: 从内存波形直接算嵌入 (用于增强样本)

In [ ]:
# ============================================================
# 流式波形加载: tf.py_function 包装 librosa
# ============================================================
def _load_wf_py(path):
    "Python 函数: 读 OGG -> 16k mono -> pad/clip 5s -> 峰值归一化。解码失败用零向量占位 (不中断整条流水线)。"
    p = path.numpy().decode("utf-8")
    target = cfg.CLIP_SAMPLES
    try:
        y, _ = librosa.load(p, sr=cfg.SR, mono=True, res_type="kaiser_fast")
    except Exception as e:
        print(f"  [WARN] 音频解码失败, 零向量占位: {p} ({type(e).__name__}: {e})")
        return np.zeros(target, dtype=np.float32)
    if len(y) == 0:
        print(f"  [WARN] 音频为空, 零向量占位: {p}")
        return np.zeros(target, dtype=np.float32)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        start = (len(y) - target) // 2
        y = y[start:start + target]
    peak = np.max(np.abs(y)) + 1e-9
    return (y / peak).astype(np.float32)

def load_waveform_tf(path):
    "tf.data wrapper: path tensor -> waveform tensor [CLIP_SAMPLES]。"
    wf = tf.py_function(_load_wf_py, [path], tf.float32)
    wf.set_shape([cfg.CLIP_SAMPLES])
    return wf

# ============================================================
# 冻结 YAMNet 嵌入提取器 (mean / mean_max / mean_max_std)
# ============================================================
def _pool_frames(frames):
    "对 (B, T, 1024) 帧嵌入按 cfg.POOL_MODE 聚合。返回 (B, D)。"
    mean = tf.reduce_mean(frames, axis=1)                # (B, 1024)
    if cfg.POOL_MODE == "mean":
        return mean
    mx = tf.reduce_max(frames, axis=1)                    # (B, 1024)
    if cfg.POOL_MODE == "mean_max":
        return tf.concat([mean, mx], axis=1)             # (B, 2048)
    std = tf.math.reduce_std(frames, axis=1)              # (B, 1024)
    return tf.concat([mean, mx, std], axis=1)            # (B, 3072)

def build_embedding_extractor():
    "构建冻结 YAMNet KerasLayer + 帧聚合 -> D 维嵌入模型。"
    yamnet_layer = hub.KerasLayer(
        cfg.YAMNET_HANDLE,
        trainable=False,
        name="yamnet_frozen",
    )

    # ---- 从 YAMNet 输出中选出 1024 维 frame_embeddings ----
    def _select_frame_emb(out):
        "从 dict/tuple/single tensor 中选 last_dim==1024 的 frame_embeddings。"
        def _is_1024(v):
            shp = getattr(v, "shape", None)
            if shp is None or len(shp) < 1:
                return False
            return int(shp[-1]) == 1024
        if isinstance(out, dict):
            for k in ("frame_embeddings", "embedding", "embeddings"):
                if k in out and _is_1024(out[k]):
                    return out[k]
            for v in out.values():
                if _is_1024(v):
                    return v
            raise ValueError(
                "YAMNet dict 输出中找不到 1024 维 frame_embeddings: "
                "keys=" + str(list(out.keys())) + ", shapes="
                + str([(k, getattr(v, "shape", None)) for k, v in out.items()])
            )
        if isinstance(out, (list, tuple)):
            for v in out:
                if _is_1024(v):
                    return v
            return out[0]
        return out

    # ---- 探测: YAMNet KerasLayer 是否接受批量 (B, T) 输入? ----
    # 批量调用可让 GPU 一次前向处理整 batch, 比 per-sample 串行快 5-10x.
    BATCHED = False
    try:
        _probe_x = tf.zeros((2, cfg.CLIP_SAMPLES), dtype=tf.float32)
        _probe_out = yamnet_layer(_probe_x)
        _probe_fe = _select_frame_emb(_probe_out)
        if _probe_fe is not None and len(_probe_fe.shape) >= 2 and int(_probe_fe.shape[0]) == 2:
            BATCHED = True
    except Exception as e:
        print(f"  [probe] 批量调用探测失败: {type(e).__name__}: {e}")
        BATCHED = False
    if BATCHED:
        print("[YAMNet] 批量调用: 支持 (单次前向处理整 batch, 最优路径)")
    else:
        print("[YAMNet] 批量调用: 不支持, 回退到 tf.map_fn(parallel_iterations=8)")

    def _yamnet_batch(x):  # x: (B, 80000) -> (B, T, 1024)
        if BATCHED:
            # 最优: 一次批量前向
            return _select_frame_emb(yamnet_layer(x))   # (B, T, 1024)
        # 回退: tf.map_fn 并行 (比 tf.vectorized_map 更高效)
        def _per_sample(w):  # w: (80000,) -> (T, 1024)
            return _select_frame_emb(yamnet_layer(w))
        return tf.map_fn(_per_sample, x, parallel_iterations=8,
                         fn_output_signature=tf.TensorSpec(
                             shape=(None, 1024), dtype=tf.float32),
                         name="yamnet_per_sample")

    inp = tf.keras.Input(shape=(cfg.CLIP_SAMPLES,), dtype=tf.float32, name="waveform")
    emb = tf.keras.layers.Lambda(_yamnet_batch, name="yamnet_call")(inp)  # (B, T, 1024)
    if len(emb.shape) == 3:
        pooled = tf.keras.layers.Lambda(_pool_frames, name="frame_pool")(emb)
    else:
        pooled = emb
    model = tf.keras.Model(inp, pooled, name="yamnet_emb_extractor")
    print(f"[YAMNet] 冻结嵌入提取器: POOL_MODE={cfg.POOL_MODE}, "
          f"输出维度={model.output_shape} (D={cfg.EMBED_DIM})")
    return model, yamnet_layer

# ============================================================
# 流式批量嵌入计算
# ============================================================
def compute_embeddings_streaming(filepaths, emb_model, batch_size=cfg.EMB_BATCH_SIZE):
    "流式计算嵌入: paths -> tf.data -> batch -> YAMNet 前向 -> (N, D)。"
    filepaths = list(filepaths)
    n = len(filepaths)
    print(f"  [emb] 流式计算 {n} 条 ...")
    if n == 0:
        return np.zeros((0, cfg.EMBED_DIM), dtype=np.float32)
    ds = tf.data.Dataset.from_tensor_slices(filepaths)
    ds = ds.map(load_waveform_tf, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    embs = []
    for i, batch in enumerate(ds):
        emb = emb_model(batch, training=False).numpy()
        embs.append(emb)
        done = min((i + 1) * batch_size, n)
        if (done % 500 < batch_size) or done == n:
            print(f"    [emb] {done}/{n} ({100*done/n:.1f}%)")
    return np.concatenate(embs, axis=0).astype(np.float32)


def compute_embeddings_from_arrays(waveforms, emb_model,
                                   batch_size=cfg.EMB_BATCH_SIZE):
    "从内存波形数组直接计算嵌入 (用于增强样本, 无文件路径)。"
    n = len(waveforms)
    if n == 0:
        return np.zeros((0, cfg.EMBED_DIM), dtype=np.float32)
    print(f"  [emb] 从内存波形计算 {n} 条增强样本嵌入 ...")
    embs = []
    for i in range(0, n, batch_size):
        batch = np.array(waveforms[i:i + batch_size], dtype=np.float32)
        emb = emb_model(batch, training=False).numpy()
        embs.append(emb)
        done = min(i + batch_size, n)
        if (done % 500 < batch_size) or done == n:
            print(f"    [emb-aug] {done}/{n} ({100*done/n:.1f}%)")
    return np.concatenate(embs, axis=0).astype(np.float32)

In [ ]:
# ============================================================
# 低资源鸟类音频实时增强模块 (内联自 audio_augmentation.py)
#
# 用途: 对样本数不足的物种进行实时音频增强,
#       无需预先生成增强文件, 每次运行随机选择增强方法。
# 依赖: librosa, numpy, hashlib (已在顶部导入)
# 用法: 必须在每折 train/val split 之后、只对本折训练集调用
#       (传入 fold 做文件名隔离), 否则增强样本会泄漏进验证集。
# ============================================================

# ── 单种增强方法 ──────────────────────────────────────────

def aug_time_stretch(y, rate):
    """时间拉伸: 变速不变调。rate<1 减速, rate>1 加速。"""
    return librosa.effects.time_stretch(y=y, rate=rate)


def aug_pitch_shift(y, sr, n_steps):
    """音高偏移: n_steps 个半音。正值升调, 负值降调。"""
    return librosa.effects.pitch_shift(y=y, sr=sr, n_steps=n_steps)


def aug_add_noise(y, snr_db, seed=None):
    """叠加高斯白噪声, 指定信噪比 (dB)。"""
    rng = np.random.RandomState(seed) if seed is not None else np.random
    signal_power = np.mean(y ** 2)
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = rng.randn(len(y)).astype(np.float32) * np.sqrt(noise_power)
    return (y + noise).astype(np.float32)


def aug_change_volume(y, gain):
    """音量缩放: gain<1 减小, gain>1 增大。"""
    return (y * gain).astype(np.float32)


# ── 增强方法调度表 ──────────────────────────────────────────

AUG_METHODS = {
    "time_stretch": {
        "fn": aug_time_stretch,
        "params": [0.85, 0.90, 0.95, 1.05, 1.10],
        "tag": lambda p: f"ts{int(p*100):03d}",
    },
    "pitch_shift": {
        "fn": aug_pitch_shift,
        "params": [-2, -1, 1, 2],
        "tag": lambda p: f"ps{p:+d}",
        "needs_sr": True,
    },
    "noise": {
        "fn": aug_add_noise,
        "params": [5, 10, 15],
        "tag": lambda p: f"noise{p}db",
        "needs_seed": True,
    },
    "volume": {
        "fn": aug_change_volume,
        "params": [0.70, 0.85, 1.15, 1.30],
        "tag": lambda p: f"vol{int(p*100):03d}",
    },
}


def augment_waveform(y, sr=16000, methods=None, seed=None):
    "对单条波形应用一次随机增强。返回 (augmented_y, method_tag)。"
    rng = np.random.RandomState(seed) if seed is not None else np.random
    method_names = list(AUG_METHODS.keys()) if methods is None else methods
    name = rng.choice(method_names)
    info = AUG_METHODS[name]
    param = rng.choice(info["params"])

    args = [y]
    if info.get("needs_sr"):
        args.append(sr)
    args.append(param)
    kwargs = {}
    if info.get("needs_seed"):
        # 派生独立子种子: 噪声波形本身也可复现 (修复: 原先噪声走全局 np.random)
        kwargs["seed"] = int(rng.randint(0, 2**31 - 1))
    aug_y = info["fn"](*args, **kwargs)

    # 确保长度一致
    if len(aug_y) != len(y):
        if len(aug_y) < len(y):
            aug_y = np.pad(aug_y, (0, len(y) - len(aug_y)))
        else:
            aug_y = aug_y[:len(y)]

    return aug_y.astype(np.float32), info["tag"](param)


# ── DataFrame 扩展: 生成增强样本的元数据行 ─────────────────

def expand_train_df(df_train, low_resource_species, target_per_species=15,
                    max_aug_per_species=50, seed=42, fold=None):
    """为低资源物种生成增强样本的 DataFrame 行 (只生成元数据, 不处理音频)。
    fold 非 None 时文件名带 fold{N}__ 前缀 + 源文件名 md5, 保证折间隔离且全局唯一
    (避免不同目录下同名文件 stem 碰撞)。"""
    df = df_train.copy()
    df["_is_augmented"] = False
    df["_original_filename"] = ""

    new_rows = []
    aug_info = []
    rng = np.random.RandomState(seed)
    prefix = f"fold{fold}__" if fold is not None else ""

    for species in low_resource_species:
        sp_mask = df["primary_label"] == species
        sp_df = df[sp_mask]
        current_count = len(sp_df)
        needed = min(target_per_species - current_count, max_aug_per_species)
        if needed <= 0:
            continue

        if needed >= current_count:
            variants_per_sample = needed // current_count
            extra = needed % current_count
            sample_variants = []
            for i in range(current_count):
                n = variants_per_sample + (1 if i < extra else 0)
                n = min(n, 16)  # 单条原始录音最多 16 个变体
                sample_variants.append(n)
        else:
            chosen_idx = set(rng.choice(current_count, needed, replace=False))
            sample_variants = [1 if i in chosen_idx else 0
                               for i in range(current_count)]

        for i, (_, row) in enumerate(sp_df.iterrows()):
            n_variants = sample_variants[i]
            if n_variants <= 0:
                continue
            original_fn = str(row["filename"])
            stem = Path(original_fn).stem
            digest = hashlib.md5(original_fn.encode("utf-8")).hexdigest()[:8]
            for v in range(n_variants):
                new_fn = f"{prefix}{stem}_{digest}_aug{v:02d}.wav"
                new_row = row.to_dict()
                new_row["filename"] = new_fn
                new_row["filepath"] = ""
                new_row["_is_augmented"] = True
                new_row["_original_filename"] = original_fn
                new_rows.append(new_row)
                aug_info.append({
                    "species": species,
                    "original_filename": original_fn,
                    "new_filename": new_fn,
                    "variant_index": v,
                    # 混入源文件哈希: 不同文件的同号变体采用不同增强 (提升多样性)
                    "seed_offset": int(digest, 16) % 2_000_000_000,
                })

    if new_rows:
        df_expanded = pd.concat([df, pd.DataFrame(new_rows)],
                                ignore_index=True)
    else:
        df_expanded = df

    return df_expanded, aug_info


# ── 缓存注入: 将增强波形加入 fn2wf ─────────────────────────

def inject_augmented_waveforms(fn2wf, aug_info, sr=16000, seed=42):
    "从原始波形生成增强变体, 以新 filename 为 key 存入 fn2wf (原地修改)。"
    by_original = {}
    for info in aug_info:
        orig = info["original_filename"]
        if orig not in by_original:
            by_original[orig] = []
        by_original[orig].append(info)

    for orig_fn, infos in by_original.items():
        if orig_fn not in fn2wf:
            print(f"  [WARN] 原始波形不在缓存中: {orig_fn}, "
                  f"跳过 {len(infos)} 条增强")
            continue
        base_wf = fn2wf[orig_fn].copy()
        for info in infos:
            aug_wf, tag = augment_waveform(
                base_wf, sr=sr,
                seed=seed + info.get("seed_offset", 0) + info["variant_index"])
            fn2wf[info["new_filename"]] = aug_wf
            info["actual_tag"] = tag


# ── 一站式入口 ────────────────────────────────────────────

def expand_with_augmentation(df_train, fn2wf, X_train, y_train_raw,
                              target_per_species=15, max_aug_per_species=50,
                              sr=16000, seed=42, fold=None):
    """一站式: 识别低资源物种 → 扩展 DataFrame → 注入增强波形 → 重建训练数组。
    fold: 传入折号用于隔离增强文件名 (跨折防泄漏)。"""
    # 1. 识别低资源物种
    per_class = df_train["primary_label"].value_counts()
    low_resource = per_class[per_class < target_per_species]

    if len(low_resource) == 0:
        print("[增强] 所有物种均 >= target_per_species, 无需增强。")
        return df_train, fn2wf, X_train, y_train_raw, {}

    print(f"[增强] 低资源物种: {len(low_resource)} 种 "
          f"(少于 {target_per_species} 条)")
    for sp, cnt in low_resource.items():
        added = min(target_per_species - cnt, max_aug_per_species)
        print(f"  {sp:>10s}: {cnt} -> {cnt + added} (+{added})")

    # 2. 扩展 DataFrame 元数据
    df_expanded, aug_info = expand_train_df(
        df_train, low_resource.index.tolist(), target_per_species,
        max_aug_per_species=max_aug_per_species, seed=seed, fold=fold)
    print(f"[增强] DataFrame: {len(df_train)} -> {len(df_expanded)} 行 "
          f"(+{len(aug_info)} 条增强)")

    # 3. 注入增强波形到缓存
    inject_augmented_waveforms(fn2wf, aug_info, sr=sr, seed=seed)
    print(f"[增强] 波形缓存: {len(fn2wf)} 条 (含增强)")

    # 4. 重建训练数组 (波形与标签同步过滤, 修复: 缺失时 X/y 错位)
    aug_mask = df_expanded["_is_augmented"] == True
    aug_fns = df_expanded[aug_mask]["filename"].tolist()
    aug_labels = df_expanded[aug_mask]["primary_label"].tolist()

    X_aug_list = []
    y_aug_list = []
    missing = 0
    for fn, lab in zip(aug_fns, aug_labels):
        if fn in fn2wf:
            X_aug_list.append(fn2wf[fn])
            y_aug_list.append(lab)
        else:
            missing += 1
    if missing:
        print(f"  [WARN] {missing} 条增强波形未找到, 已跳过")

    if X_aug_list:
        X_aug = np.stack(X_aug_list).astype(np.float32)
        y_aug = np.array(y_aug_list)
        X_expanded = np.concatenate([X_train, X_aug], axis=0)
        y_expanded = np.concatenate([y_train_raw, y_aug], axis=0)
    else:
        X_expanded = X_train
        y_expanded = y_train_raw

    print(f"[增强] X_train: {X_train.shape} -> {X_expanded.shape}")

    # 5. 生成报告
    report = {}
    for sp in low_resource.index:
        before = int(low_resource[sp])
        after = int((df_expanded["primary_label"] == sp).sum())
        report[sp] = {"before": before, "after": after,
                      "added": after - before}

    return df_expanded, fn2wf, X_expanded, y_expanded, report

## Cell 4/7: 嵌入缓存 + 折内低资源增强 + LightGBM 导出 + 5 折分类头训练

流程:
1. 一次性扫描全部 CSV, 收集唯一 (filename, filepath, label), 流式算嵌入
2. 缓存到 embeddings_all.npz (~590 MB, 跨折复用)
3. 契约导出 embeddings.npz (仅原始样本; 增强样本按折写入 fold{N}_train.npz)
4. 每折: **折内低资源增强** (仅用本折训练集, 杜绝验证泄漏) -> 切片 train/val/test 嵌入 -> 训练 Dense 头 -> 综合评估 -> 导出 LightGBM .npz (train 含增强)

In [ ]:
# ============================================================
# 一次性嵌入缓存 (全数据集唯一文件, 内部用)
# ============================================================
def build_embedding_cache(csv_paths, year2root, all_roots, emb_model):
    "一次性为全部 CSV 中出现的唯一文件计算嵌入, 缓存到 .npz。"
    cache_path = cfg.EMB_CACHE
    if Path(cache_path).exists():
        d = np.load(cache_path, allow_pickle=True)
        cached_dim = int(d["emb"].shape[1]) if d["emb"].ndim == 2 else -1
        if cached_dim != cfg.EMBED_DIM:
            print(f"[缓存] 维度不匹配 (cached={cached_dim} vs cfg.EMBED_DIM={cfg.EMBED_DIM}), "
                  f"删除旧缓存并重算: {cache_path}")
            os.remove(cache_path)
        else:
            print(f"[缓存] 读取: {cache_path}")
            fn2emb = {str(fn): d["emb"][i] for i, fn in enumerate(d["filenames"])}
            fn2label = {str(fn): str(lab) for i, (fn, lab)
                        in enumerate(zip(d["filenames"], d["labels_str"]))}
            print(f"  缓存命中 {len(fn2emb)} 条, dim={cached_dim}")
            return fn2emb, fn2label


    fn2path = {}
    fn2label = {}
    csv_files = [cfg.TRAIN_WEIGHTED_CSV, cfg.TEST_CSV]
    for f in range(1, cfg.N_FOLDS + 1):
        csv_files.extend(cfg.fold_csvs(f))

    for fname in csv_files:
        if fname not in csv_paths:
            continue
        df = pd.read_csv(csv_paths[fname])
        for row in df.to_dict('records'):
            fn = str(row["filename"])
            if fn in fn2path:
                continue
            p = resolve_audio_path(row, year2root, all_roots)
            if p is not None:
                fn2path[fn] = str(p)
                fn2label[fn] = str(row["primary_label"])

    fn_list = list(fn2path.keys())
    path_list = [fn2path[fn] for fn in fn_list]
    label_list = [fn2label[fn] for fn in fn_list]
    print(f"[缓存] {len(fn_list)} 个唯一文件, 开始流式计算嵌入 ...")
    print(f"  预计耗时 ~{len(fn_list)//400//60}h{len(fn_list)//400%60}m "
          f"(按 400 条/分钟估算)")

    X = compute_embeddings_streaming(path_list, emb_model)

    # 维度契约: 必须 == cfg.EMBED_DIM (1024/2048/3072).
    # 若为 521/1563, 说明 _per_sample 误取了 logits 而非 frame_embeddings.
    if X.ndim != 2 or X.shape[1] != cfg.EMBED_DIM:
        raise SystemExit(
            f"[契约失败] 嵌入维度 X.shape={X.shape} 与 cfg.EMBED_DIM={cfg.EMBED_DIM} 不符. "
            f"若 dim=521 或 1563, 说明 YAMNet 返回了 logits 而非 frame_embeddings; "
            f"请检查 _per_sample 的输出选择逻辑."
        )

    np.savez(cache_path,
             filenames=np.array(fn_list),
             labels_str=np.array(label_list),
             emb=X)
    print(f"[缓存] 已存: {cache_path} ({X.shape}, {X.nbytes/1e9:.2f} GB)")
    fn2emb = {fn: X[i] for i, fn in enumerate(fn_list)}
    return fn2emb, fn2label


# ============================================================
# 类频率统计 (仅用于分层评估, 不参与增强)
# ============================================================
def compute_class_train_counts(csv_paths):
    "读取完整训练 CSV, 返回 {species: count}。仅用于评估时的类频率分层。"
    if cfg.TRAIN_WEIGHTED_CSV not in csv_paths:
        print("[统计] 训练 CSV 不存在, 类频率统计为空")
        return {}
    df = pd.read_csv(csv_paths[cfg.TRAIN_WEIGHTED_CSV], usecols=["primary_label"])
    return df["primary_label"].value_counts().to_dict()


# ============================================================
# 折内低资源物种增强 (修复跨折泄漏: 只使用本折训练集)
# ============================================================
def load_waveform_local(p):
    "读取音频 -> 16k mono -> pad/clip 5s -> 峰值归一化。失败返回 None (不中断流程)。"
    try:
        y, _ = librosa.load(p, sr=cfg.SR, mono=True, res_type="kaiser_fast")
    except Exception as e:
        print(f"  [WARN] 音频解码失败, 跳过: {p} ({type(e).__name__}: {e})")
        return None
    if len(y) == 0:
        print(f"  [WARN] 音频为空, 跳过: {p}")
        return None
    target = cfg.CLIP_SAMPLES
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        s = (len(y) - target) // 2
        y = y[s:s + target]
    peak = np.max(np.abs(y)) + 1e-9
    return (y / peak).astype(np.float32)


def augment_fold_train(fold, df_train, year2root, all_roots, emb_model,
                       fn2emb, fn2label, fn2wf_cache):
    """折内增强: 仅对本折训练集中的低资源物种做增强, 杜绝验证集泄漏。
    增强嵌入以折隔离的 filename (fold{N}__..._augNN.wav) 原地注入 fn2emb/fn2label;
    fn2wf_cache 跨折复用原始波形解码。返回扩展后的 df_train (含增强行)。"""
    class_counts = df_train["primary_label"].value_counts().to_dict()
    low_resource = [sp for sp, cnt in class_counts.items()
                    if cnt < cfg.LOW_RESOURCE_THRESHOLD]
    if not low_resource:
        print(f"[增强] fold{fold}: 无低资源物种 (<{cfg.LOW_RESOURCE_THRESHOLD}), 跳过")
        return df_train

    lr_df = df_train[df_train["primary_label"].isin(low_resource)].copy()

    # 加载低资源波形 (跨折缓存复用; 解码失败自动跳过)
    n_loaded = 0
    for _, row in lr_df.drop_duplicates("filename").iterrows():
        fn = str(row["filename"])
        if fn in fn2wf_cache:
            continue
        p = resolve_audio_path(row, year2root, all_roots)
        if p is None:
            print(f"  [WARN] fold{fold}: 找不到音频, 跳过: {fn}")
            continue
        wf = load_waveform_local(p)
        if wf is not None:
            fn2wf_cache[fn] = wf
            n_loaded += 1

    lr_df = lr_df[lr_df["filename"].astype(str).isin(fn2wf_cache)].reset_index(drop=True)
    if len(lr_df) == 0:
        print(f"[增强] fold{fold}: 无可用低资源波形, 跳过")
        return df_train
    print(f"[增强] fold{fold}: {len(low_resource)} 个低资源物种, "
          f"波形 {len(lr_df)} 条 (新加载 {n_loaded}, 缓存命中 {len(lr_df) - n_loaded})")

    X_lr = np.stack([fn2wf_cache[fn] for fn in lr_df["filename"].astype(str)]).astype(np.float32)
    y_lr = lr_df["primary_label"].values.astype(str)

    # 本折专属波形字典 (浅拷贝): 增强波形不污染跨折缓存, 本折结束后即释放
    fn2wf_fold = {fn: fn2wf_cache[fn] for fn in lr_df["filename"].astype(str)}

    df_expanded, fn2wf_fold, X_expanded, y_expanded, report = expand_with_augmentation(
        lr_df, fn2wf_fold, X_lr, y_lr,
        target_per_species=cfg.AUG_TARGET_PER_SPECIES,
        max_aug_per_species=cfg.AUG_MAX_PER_SPECIES,
        sr=cfg.SR, seed=cfg.SEED + fold, fold=fold)

    orig_fns = set(lr_df["filename"].astype(str))
    df_aug = df_expanded[~df_expanded["filename"].astype(str).isin(orig_fns)].reset_index(drop=True)
    if len(df_aug) == 0:
        print(f"[增强] fold{fold}: 未生成增强样本")
        return df_train

    aug_fns = df_aug["filename"].astype(str).tolist()
    valid_fns = [fn for fn in aug_fns if fn in fn2wf_fold]
    aug_waveforms = [fn2wf_fold[fn] for fn in valid_fns]
    if not aug_waveforms:
        print(f"[增强] fold{fold}: 无有效增强波形, 跳过")
        return df_train

    X_aug_emb = compute_embeddings_from_arrays(aug_waveforms, emb_model)

    aug_label = dict(zip(df_aug["filename"].astype(str),
                         df_aug["primary_label"].astype(str)))
    for i, fn in enumerate(valid_fns):
        fn2emb[fn] = X_aug_emb[i]
        fn2label[fn] = aug_label[fn]

    df_aug = df_aug[df_aug["filename"].astype(str).isin(valid_fns)].reset_index(drop=True)
    print(f"[增强] fold{fold}: +{len(df_aug)} 条增强样本 "
          f"(覆盖 {df_aug['primary_label'].nunique()} 个物种)")
    return pd.concat([df_train, df_aug], ignore_index=True)


# ============================================================
# 契约导出: yamnet_embeddings/embeddings.npz (filenames + emb)
# ============================================================
def export_contract_embeddings(fn2emb, fn2label):
    "按队友契约导出: yamnet_embeddings/embeddings.npz, key=filenames+emb (仅原始样本)。"
    fn_list = list(fn2emb.keys())
    emb_arr = np.stack([fn2emb[fn] for fn in fn_list]).astype(np.float32)
    out_path = os.path.join(cfg.EMB_DIR, "embeddings.npz")
    np.savez(out_path,
             filenames=np.array(fn_list),
             emb=emb_arr)
    print(f"[契约] {out_path}: {len(fn_list)} 条, emb shape={emb_arr.shape}")


# ============================================================
# 嵌入切片 (按 CSV 行顺序)
# ============================================================
def slice_embeddings(fn2emb, label2idx, df):
    "按 df 的 filename 切片缓存嵌入, 跳过不在缓存的行。"
    keep_mask = np.array([str(fn) in fn2emb for fn in df["filename"]], dtype=bool)
    df_keep = df[keep_mask].reset_index(drop=True)
    X = np.stack([fn2emb[str(fn)] for fn in df_keep["filename"]]).astype(np.float32)
    y_str = np.array([str(l) for l in df_keep["primary_label"]])
    y_idx = np.array([label2idx[str(l)] for l in df_keep["primary_label"]],
                     dtype=np.int64)
    return X, y_str, y_idx, df_keep


# ============================================================
# LightGBM 嵌入导出 (额外, 含类权重)
# ============================================================
def export_for_lightgbm(fold, df_train, df_val, df_test,
                        fn2emb, label2idx, idx2label, classes):
    "每折导出 .npz: embeddings + labels_str + labels_idx + filenames + 类权重。"
    X_tr, y_tr_str, y_tr, df_tr = slice_embeddings(fn2emb, label2idx, df_train)
    X_va, y_va_str, y_va, df_va = slice_embeddings(fn2emb, label2idx, df_val)

    train_payload = {
        "embeddings": X_tr, "labels_str": y_tr_str, "labels_idx": y_tr,
        "filenames": df_tr["filename"].values,
    }
    if "loss_class_weight" in df_tr.columns:
        train_payload["loss_class_weight"] = df_tr["loss_class_weight"].values.astype(np.float32)
    if "sampler_weight" in df_tr.columns:
        train_payload["sampler_weight"] = df_tr["sampler_weight"].values.astype(np.float32)
    np.savez(os.path.join(cfg.EMB_DIR, f"fold{fold}_train.npz"), **train_payload)

    np.savez(os.path.join(cfg.EMB_DIR, f"fold{fold}_val.npz"),
             embeddings=X_va, labels_str=y_va_str, labels_idx=y_va,
             filenames=df_va["filename"].values)

    if fold == 1:
        X_te, y_te_str, y_te, df_te = slice_embeddings(fn2emb, label2idx, df_test)
        np.savez(os.path.join(cfg.EMB_DIR, "test.npz"),
                 embeddings=X_te, labels_str=y_te_str, labels_idx=y_te,
                 filenames=df_te["filename"].values)
        with open(os.path.join(cfg.EMB_DIR, "label_map.json"), "w", encoding="utf-8") as f:
            json.dump({"label2idx": label2idx,
                       "idx2label": {str(k): v for k, v in idx2label.items()},
                       "classes": classes}, f, ensure_ascii=False, indent=2)

    print(f"[导出] fold{fold}: train={X_tr.shape} val={X_va.shape} "
          f"-> {cfg.EMB_DIR}/fold{fold}_*.npz")


# ============================================================
# 综合评估指标
# ============================================================
def compute_all_metrics(y_true, y_pred, y_prob, num_classes,
                        class_train_counts=None, label2idx=None, top_k=5):
    "计算 Top-1, Top-5, macro-F1, balanced accuracy + 类频率分层。"
    metrics = {
        "top1_acc": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro",
                                   zero_division=0)),
        "balanced_acc": float(balanced_accuracy_score(y_true, y_pred)),
    }
    # Top-5
    if num_classes > top_k and y_prob is not None:
        try:
            metrics["top5_acc"] = float(top_k_accuracy_score(
                y_true, y_prob, k=top_k, labels=list(range(num_classes))))
        except Exception:
            metrics["top5_acc"] = float("nan")
    else:
        metrics["top5_acc"] = metrics["top1_acc"]

    # 按类频率分组分层评估
    if class_train_counts and label2idx:
        idx2count = {}
        for cls_name, count in class_train_counts.items():
            if cls_name in label2idx:
                idx2count[label2idx[cls_name]] = count
        for gi in range(len(cfg.FREQ_GROUP_BINS) - 1):
            lo, hi = cfg.FREQ_GROUP_BINS[gi], cfg.FREQ_GROUP_BINS[gi + 1]
            gname = cfg.FREQ_GROUP_NAMES[gi]
            mask = np.array([lo <= idx2count.get(int(t), 0) < hi
                             for t in y_true])
            if mask.sum() > 0:
                metrics[f"acc_{gname}"] = float(
                    accuracy_score(y_true[mask], y_pred[mask]))
                metrics[f"f1_{gname}"] = float(f1_score(
                    y_true[mask], y_pred[mask], average="macro",
                    zero_division=0))
            else:
                metrics[f"acc_{gname}"] = float("nan")
                metrics[f"f1_{gname}"] = float("nan")
    return metrics


# ============================================================
# Dense 分类头
# ============================================================
def build_classifier(num_classes, embedding_dim=cfg.EMBED_DIM,
                     dropout=cfg.DROPOUT):
    "embedding_dim 维嵌入 -> Dense 256 -> Dropout -> softmax。"
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(embedding_dim,)),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dropout(dropout),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ], name="yamnet_head")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(cfg.HEAD_LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# ============================================================
# 5 折 CV: 折内低资源增强 + 训练 Dense 头 + 导出 LightGBM 嵌入
# ============================================================
def main_cv_head(fn2emb, fn2label, classes, label2idx, idx2label, csv_paths,
                 year2root, all_roots, emb_model, class_train_counts=None):
    clean_results = []
    fn2wf_cache = {}   # 跨折复用低资源原始波形解码
    for fold in range(1, cfg.N_FOLDS + 1):
        print(f"\n{'#'*20} FOLD {fold}/{cfg.N_FOLDS} {'#'*20}")
        tf.random.set_seed(cfg.SEED + fold)
        np.random.seed(cfg.SEED + fold)

        tr_csv, va_csv = cfg.fold_csvs(fold)
        df_train = pd.read_csv(csv_paths[tr_csv])
        df_val = pd.read_csv(csv_paths[va_csv])
        df_test = pd.read_csv(csv_paths[cfg.TEST_CSV])

        # 折内低资源增强: 只用本折训练集, 杜绝验证集泄漏
        df_train = augment_fold_train(fold, df_train, year2root, all_roots,
                                      emb_model, fn2emb, fn2label, fn2wf_cache)

        print(f"[CSV] train={len(df_train)}(含增强) val={len(df_val)} test={len(df_test)}")

        X_tr, y_tr_str, y_tr, df_tr = slice_embeddings(fn2emb, label2idx, df_train)
        X_va, y_va_str, y_va, df_va = slice_embeddings(fn2emb, label2idx, df_val)
        X_te, y_te_str, y_te, df_te = slice_embeddings(fn2emb, label2idx, df_test)
        print(f"[切片] train={X_tr.shape} val={X_va.shape} test={X_te.shape}")

        export_for_lightgbm(fold, df_tr, df_va, df_te,
                            fn2emb, label2idx, idx2label, classes)

        fold_out = Path(cfg.OUT_DIR) / f"fold{fold}"
        fold_out.mkdir(parents=True, exist_ok=True)
        model = build_classifier(num_classes=len(classes))
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_accuracy", patience=8, restore_best_weights=True),
            tf.keras.callbacks.ModelCheckpoint(
                fold_out / "head_model.keras", save_best_only=True,
                monitor="val_accuracy"),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
        ]

        # 训练集 sample_weight (loss_class_weight); 验证集不加权
        if "loss_class_weight" in df_tr.columns:
            sw_tr = df_tr["loss_class_weight"].values.astype(np.float32)
        else:
            sw_tr = None

        print(f"[训练] fold{fold} 开始 ...")
        model.fit(X_tr, y_tr,
                  validation_data=(X_va, y_va),
                  sample_weight=sw_tr,
                  epochs=cfg.HEAD_EPOCHS,
                  batch_size=cfg.HEAD_BATCH_SIZE,
                  callbacks=callbacks, verbose=2)

        # ---- 综合评估 (测试集) ----
        y_prob = model.predict(X_te, verbose=0)
        y_pred = np.argmax(y_prob, axis=1)
        fold_metrics = compute_all_metrics(
            y_te, y_pred, y_prob, len(classes),
            class_train_counts=class_train_counts, label2idx=label2idx)
        print(f"[结果] fold{fold}: " +
              "  ".join(f"{k}={v:.4f}" for k, v in fold_metrics.items()))
        clean_results.append(fold_metrics)

        np.savez(fold_out / "test_predictions.npz",
                 y_true=y_te, y_pred=y_pred, y_prob=y_prob,
                 classes=np.array(classes),
                 test_filenames=df_te["filename"].values)
        print(f"[结果] fold{fold} 已存: {fold_out}")

    # ---- 汇总 5 折 ----
    metrics_df = pd.DataFrame(clean_results)
    metrics_df.insert(0, "fold", range(1, cfg.N_FOLDS + 1))
    metrics_df.to_csv(os.path.join(cfg.OUT_DIR, "cv_per_fold.csv"), index=False)

    summary_rows = []
    for col in metrics_df.columns:
        if col == "fold":
            continue
        vals = metrics_df[col].dropna()
        if len(vals) > 0:
            summary_rows.append({"metric": col,
                                 "mean": float(vals.mean()),
                                 "std": float(vals.std(ddof=0))})
    pd.DataFrame(summary_rows).to_csv(
        os.path.join(cfg.OUT_DIR, "cv_summary.csv"), index=False)

    print(f"\n[CV 汇总]")
    for r in summary_rows:
        print(f"  {r['metric']}: {r['mean']:.4f} +/- {r['std']:.4f}")
    print(f"[CV] LightGBM 嵌入已导出到: {cfg.EMB_DIR}")
    return clean_results

## Cell 5/7: 噪声鲁棒性评估 (流式)

对测试集干净音频叠加 3 档高斯白噪声 (5dB / 0dB / -5dB),
重新流式算嵌入, 用各折 Dense 头预测。
每档 SNR 计算综合指标: Top-1, Top-5, macro-F1, balanced accuracy, 类频率分层。
汇总 mean +/- std 写入 cv_summary.csv。

In [ ]:
SNR_TIERS = ["clean", "5dB", "0dB", "-5dB"]


def add_gaussian_noise(wf, snr_db, rng):
    "按指定 SNR 叠加高斯白噪声, 叠后重新归一化到 [-1,1]。"
    signal_power = np.mean(wf ** 2) + 1e-12
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = rng.normal(0, np.sqrt(noise_power), size=wf.shape).astype(np.float32)
    noisy = wf + noise
    peak = np.max(np.abs(noisy)) + 1e-9
    return (noisy / peak).astype(np.float32)


def _load_noisy_wf_py(path, snr, seed):
    "Python 函数: 读 OGG -> 16k mono -> pad/clip -> 归一化 -> 叠噪。解码失败用零向量占位。"
    p = path.numpy().decode("utf-8")
    snr_val = float(snr.numpy())
    seed_val = int(seed.numpy())
    target = cfg.CLIP_SAMPLES
    try:
        y, _ = librosa.load(p, sr=cfg.SR, mono=True, res_type="kaiser_fast")
    except Exception as e:
        print(f"  [WARN] 音频解码失败, 零向量占位: {p} ({type(e).__name__}: {e})")
        return np.zeros(target, dtype=np.float32)
    if len(y) == 0:
        print(f"  [WARN] 音频为空, 零向量占位: {p}")
        return np.zeros(target, dtype=np.float32)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        s = (len(y) - target) // 2
        y = y[s:s + target]
    peak = np.max(np.abs(y)) + 1e-9
    clean = (y / peak).astype(np.float32)
    rng = np.random.default_rng(seed_val)
    return add_gaussian_noise(clean, snr_val, rng)


def load_noisy_waveform_tf(path, snr, seed):
    "tf.data wrapper: (path, snr, seed) -> noisy waveform [CLIP_SAMPLES]。"
    wf = tf.py_function(_load_noisy_wf_py, [path, snr, seed], tf.float32)
    wf.set_shape([cfg.CLIP_SAMPLES])
    return wf


def build_noise_embeddings_streaming(df_test, year2root, all_roots,
                                     emb_model, fn2emb, label2idx):
    "流式计算测试集 3 档噪声嵌入; clean 直接复用主缓存。"
    cache_path = cfg.NOISE_EMB_CACHE
    if Path(cache_path).exists():
        print(f"[噪声嵌入] 读缓存: {cache_path}")
        d = np.load(cache_path, allow_pickle=True)
        noise_embs = {k: d[k] for k in d.files}
        # 缓存命中也要保证契约文件存在 (可能已被清理)
        fns = noise_embs.get("test_filenames")
        for tier_file, arr_key in [("embeddings_clean.npz", "X_clean"),
                                   ("embeddings_5db.npz", "X_5db"),
                                   ("embeddings_0db.npz", "X_0db"),
                                   ("embeddings_m5db.npz", "X_m5db")]:
            out_p = os.path.join(cfg.EMB_DIR, tier_file)
            arr = noise_embs.get(arr_key)
            if arr is not None and fns is not None and not Path(out_p).exists():
                np.savez(out_p, filenames=fns, emb=arr)
                print(f"[契约] 补齐 {tier_file}: {arr.shape}")
        return noise_embs

    keep_rows = []
    keep_paths = []
    for _, row in df_test.iterrows():
        if str(row["filename"]) not in fn2emb:
            continue
        p = resolve_audio_path(row, year2root, all_roots)
        if p is not None:
            keep_rows.append(row)
            keep_paths.append(str(p))
    df_t = pd.DataFrame(keep_rows).reset_index(drop=True)
    n = len(keep_paths)
    print(f"[噪声嵌入] 测试集 {n} 条 (剔除 {len(df_test)-n} 条)")
    if n == 0:
        print("[噪声嵌入] 无可用样本, 跳过")
        return {}

    X_clean = np.stack([fn2emb[str(fn)] for fn in df_t["filename"]]).astype(np.float32)
    y_test = np.array([label2idx[str(l)] for l in df_t["primary_label"]], dtype=np.int64)
    test_filenames = df_t["filename"].values

    # 契约导出: clean 档 (测试集)
    np.savez(os.path.join(cfg.EMB_DIR, "embeddings_clean.npz"),
             filenames=test_filenames, emb=X_clean)
    print(f"[契约] embeddings_clean.npz: {n} 条, {X_clean.shape}")

    snr_tiers = [("5db", 5.0), ("0db", 0.0), ("m5db", -5.0)]
    noise_embs = {"X_clean": X_clean, "y_true": y_test,
                  "test_filenames": test_filenames}

    paths_arr = np.array(keep_paths)
    seeds_arr = np.arange(n, dtype=np.int32) + cfg.SEED

    for tier_name, snr_val in snr_tiers:
        print(f"  [噪声嵌入] tier {tier_name} (SNR={snr_val}) ...")
        snrs_arr = np.full(n, snr_val, dtype=np.float32)
        ds = tf.data.Dataset.from_tensor_slices((paths_arr, snrs_arr, seeds_arr))
        ds = ds.map(load_noisy_waveform_tf, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(cfg.EMB_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

        embs = []
        for i, batch in enumerate(ds):
            emb = emb_model(batch, training=False).numpy()
            embs.append(emb)
            done = min((i + 1) * cfg.EMB_BATCH_SIZE, n)
            if (done % 500 < cfg.EMB_BATCH_SIZE) or done == n:
                print(f"    {tier_name}: {done}/{n} ({100*done/n:.1f}%)")
        X_tier = np.concatenate(embs, axis=0).astype(np.float32)
        noise_embs[f"X_{tier_name}"] = X_tier

        # 契约导出: embeddings_5db.npz / embeddings_0db.npz / embeddings_m5db.npz
        np.savez(os.path.join(cfg.EMB_DIR, f"embeddings_{tier_name}.npz"),
                 filenames=test_filenames, emb=X_tier)
        print(f"[契约] embeddings_{tier_name}.npz: {n} 条, {X_tier.shape}")

    np.savez(cache_path, **noise_embs)
    print(f"[噪声嵌入] 内部缓存已存: {cache_path}")
    return noise_embs


def main_noise_eval(fn2emb, label2idx, idx2label, classes,
                    csv_paths, year2root, all_roots, emb_model,
                    class_train_counts=None):
    "5 折噪声评估: 流式算噪声嵌入, 各折模型复用预测, 综合指标。"
    print("\n" + "=" * 60)
    print("  阶段 2/3: 噪声鲁棒性评估 (流式)")
    print("=" * 60)

    df_test = pd.read_csv(csv_paths[cfg.TEST_CSV])
    noise_emb = build_noise_embeddings_streaming(
        df_test, year2root, all_roots, emb_model, fn2emb, label2idx)

    rows = []
    for fold in range(1, cfg.N_FOLDS + 1):
        model_path = Path(cfg.OUT_DIR) / f"fold{fold}" / "head_model.keras"
        if not model_path.exists():
            print(f"[噪声] fold{fold} 模型不存在, 跳过")
            continue
        model = tf.keras.models.load_model(model_path)
        y_true = noise_emb["y_true"]

        fold_row = {"fold": fold}
        preds_by_snr = {}
        for snr_key, arr_key in [("clean", "X_clean"), ("5dB", "X_5db"),
                                 ("0dB", "X_0db"), ("-5dB", "X_m5db")]:
            y_prob = model.predict(noise_emb[arr_key], verbose=0,
                                   batch_size=cfg.HEAD_BATCH_SIZE)
            y_pred = np.argmax(y_prob, axis=1)
            preds_by_snr[snr_key] = y_pred

            # 综合指标
            m = compute_all_metrics(
                y_true, y_pred, y_prob, len(classes),
                class_train_counts=class_train_counts,
                label2idx=label2idx)
            for mk, mv in m.items():
                fold_row[f"{snr_key}_{mk}"] = mv

        print(f"[噪声] fold{fold}: " +
              "  ".join(f"{k}_top1={fold_row[f'{k}_top1_acc']:.4f}"
                        for k in SNR_TIERS))

        np.savez(Path(cfg.OUT_DIR) / f"fold{fold}" / "noise_results.npz",
                 snr_tiers=np.array(SNR_TIERS),
                 y_true=y_true,
                 preds_clean=preds_by_snr["clean"],
                 preds_5dB=preds_by_snr["5dB"],
                 preds_0dB=preds_by_snr["0dB"],
                 preds_n5dB=preds_by_snr["-5dB"])
        rows.append(fold_row)

    if not rows:
        print("[噪声] 无可用模型, 退出")
        return

    per_fold = pd.DataFrame(rows)
    per_fold.to_csv(os.path.join(cfg.OUT_DIR, "cv_noise_per_fold.csv"),
                    index=False)

    # 汇总到 cv_summary.csv (幂等: 同名指标覆盖, 重复运行不产生重复行)
    summary_path = Path(cfg.OUT_DIR) / "cv_summary.csv"
    existing = pd.read_csv(summary_path) if summary_path.exists() else None
    if existing is None:
        existing = pd.DataFrame(columns=["metric", "mean", "std"])

    noise_metric_cols = [c for c in per_fold.columns if c != "fold"]
    new_rows = pd.DataFrame([{
        "metric": col,
        "mean": float(per_fold[col].dropna().mean()),
        "std": float(per_fold[col].dropna().std(ddof=0)),
    } for col in noise_metric_cols if per_fold[col].notna().any()])
    existing = existing[~existing["metric"].isin(new_rows["metric"])]
    pd.concat([existing, new_rows], ignore_index=True).to_csv(
        summary_path, index=False)

    print(f"\n[噪声] 汇总 (mean+/-std):")
    for _, r in new_rows.iterrows():
        print(f"  {r['metric']}: {r['mean']:.4f} +/- {r['std']:.4f}")

## Cell 6/7: 推理速度测试

测量两条路径延迟:
- Head-only: 1024 维嵌入 -> Dense 头预测
- Full pipeline: OGG 路径 -> librosa 解码 -> YAMNet 前向 -> Dense 头

In [ ]:
def measure_inference(fn2emb, label2idx, csv_paths, year2root, all_roots,
                     emb_model, n_samples=50, n_warmup=5):
    "测量推理延迟 (head-only + full pipeline), 5 折取均值。"
    print("\n" + "=" * 60)
    print("  阶段 3/3: 推理速度测试")
    print("=" * 60)

    df_test = pd.read_csv(csv_paths[cfg.TEST_CSV])
    samples = []
    for _, row in df_test.iterrows():
        if str(row["filename"]) in fn2emb:
            p = resolve_audio_path(row, year2root, all_roots)
            if p is not None:
                samples.append((str(row["filename"]), str(p)))
        if len(samples) >= n_samples:
            break
    n = len(samples)
    print(f"[推理] 取 {n} 条样本测试")

    sample_embs = np.stack([fn2emb[fn] for fn, _ in samples]).astype(np.float32)
    sample_paths = [p for _, p in samples]

    # Head-only 延迟 (5 折均值)
    head_latencies = []
    for fold in range(1, cfg.N_FOLDS + 1):
        model_path = Path(cfg.OUT_DIR) / f"fold{fold}" / "head_model.keras"
        if not model_path.exists():
            continue
        model = tf.keras.models.load_model(model_path)
        for _ in range(n_warmup):
            model.predict(sample_embs[:1], verbose=0)
        t0 = time.perf_counter()
        for i in range(n):
            model.predict(sample_embs[i:i+1], verbose=0)
        ms = (time.perf_counter() - t0) / n * 1000
        head_latencies.append(ms)
        print(f"  fold{fold} head-only: {ms:.2f} ms/sample")

    # Full pipeline 延迟 (用 fold1 模型)
    full_latencies = []
    model_path = Path(cfg.OUT_DIR) / "fold1" / "head_model.keras"
    if model_path.exists():
        model = tf.keras.models.load_model(model_path)
        for _ in range(n_warmup):
            wf = _load_wf_py(tf.constant(sample_paths[0]))
            emb = emb_model(wf[None, :], training=False)
            model.predict(emb, verbose=0)
        for i in range(n):
            t0 = time.perf_counter()
            wf = _load_wf_py(tf.constant(sample_paths[i]))
            emb = emb_model(wf[None, :], training=False)
            model.predict(emb, verbose=0)
            full_latencies.append((time.perf_counter() - t0) * 1000)
        print(f"  full pipeline mean: {np.mean(full_latencies):.2f} ms/sample")
        print(f"    (decode+YAMNet: {np.mean(full_latencies)-np.mean(head_latencies):.2f} ms, "
              f"head: {np.mean(head_latencies):.2f} ms)")

    metrics = pd.DataFrame([{
        "n_samples": n,
        "head_latency_ms_mean": float(np.mean(head_latencies)) if head_latencies else 0,
        "head_latency_ms_std": float(np.std(head_latencies)) if head_latencies else 0,
        "full_latency_ms_mean": float(np.mean(full_latencies)) if full_latencies else 0,
        "full_latency_ms_std": float(np.std(full_latencies)) if full_latencies else 0,
    }])
    metrics.to_csv(os.path.join(cfg.OUT_DIR, "inference_metrics.csv"), index=False)
    print(f"[推理] 结果已存: {cfg.OUT_DIR}/inference_metrics.csv")

## 一键运行: 嵌入提取 -> 5 折训练 -> 噪声评估 -> 推理测试

顺序不可调换: 缓存/标签映射要先于训练/噪声测试。

In [ ]:
def main():
    "顺序执行: 扫描 -> 标签 -> 嵌入 -> 契约导出 -> 5 折训练(折内增强) -> 噪声 -> 推理。"
    t0 = time.time()

    # 阶段 0: 扫描 + 标签
    print("\n" + "#" * 60)
    print("# 阶段 0/4: Kaggle 挂载扫描 + 标签映射")
    print("#" * 60 + "\n")
    print_mounted_inputs(cfg.INPUT_BASE)
    year2root, unknown_roots, csv_paths = scan_kaggle_inputs(cfg.INPUT_BASE)
    all_roots = list(year2root.values())
    print(f"[音频根] {len(year2root)} 个年度根:")
    for y, r in sorted(year2root.items(), key=lambda kv: str(kv[0])):
        print(f"  {y} -> {r}")
    if not year2root:
        raise SystemExit("[错误] 未发现 train_audio/train_short_audio 目录。")
    print(f"[CSV] 找到 {len(csv_paths)} 个 CSV:")
    for f, p in sorted(csv_paths.items()):
        print(f"  {f} -> {p}")

    classes, label2idx, idx2label = build_label_map(csv_paths)

    # 阶段 1: 嵌入提取 (一次性)
    print("\n" + "#" * 60)
    print("# 阶段 1/4: 冻结 YAMNet 嵌入提取 (流式, 一次性)")
    print("#" * 60 + "\n")
    emb_model, yamnet_layer = build_embedding_extractor()
    fn2emb, fn2label = build_embedding_cache(csv_paths, year2root, all_roots, emb_model)
    print(f"[嵌入] 共 {len(fn2emb)} 条, 维度 {next(iter(fn2emb.values())).shape}")

    # 阶段 1.5: 类频率统计 (仅用于分层评估; 低资源增强移到折内, 杜绝验证泄漏)
    print("\n" + "#" * 60)
    print("# 阶段 1.5/4: 类频率统计 (低资源增强在每折训练集内独立进行)")
    print("#" * 60 + "\n")
    class_train_counts = compute_class_train_counts(csv_paths)
    n_low = sum(1 for c in class_train_counts.values()
                if c < cfg.LOW_RESOURCE_THRESHOLD)
    print(f"[统计] {len(class_train_counts)} 个训练物种, 其中 {n_low} 个低资源 "
          f"(<{cfg.LOW_RESOURCE_THRESHOLD} 条, 将在各折训练集内独立增强)")

    # 契约导出: yamnet_embeddings/embeddings.npz (仅原始样本; 增强按折写入 fold{N}_train.npz)
    export_contract_embeddings(fn2emb, fn2label)

    # 阶段 2: 5 折训练 (折内增强) + LightGBM 导出
    print("\n" + "#" * 60)
    print("# 阶段 2/4: 5 折 Dense 头训练 (折内低资源增强) + LightGBM 嵌入导出")
    print("#" * 60 + "\n")
    main_cv_head(fn2emb, fn2label, classes, label2idx, idx2label, csv_paths,
                 year2root, all_roots, emb_model,
                 class_train_counts=class_train_counts)

    # 阶段 3: 噪声评估 (含契约导出: embeddings_5db/0db/m5db.npz)
    main_noise_eval(fn2emb, label2idx, idx2label, classes,
                    csv_paths, year2root, all_roots, emb_model,
                    class_train_counts=class_train_counts)

    # 阶段 4: 推理速度
    measure_inference(fn2emb, label2idx, csv_paths, year2root, all_roots, emb_model)

    elapsed = (time.time() - t0) / 60
    print(f"\n{'='*60}")
    print(f"[完成] 全流程耗时 {elapsed:.1f} 分钟")
    print(f"[输出] {cfg.OUT_DIR}")
    print(f"  - cv_per_fold.csv          5 折综合指标 (Top-1/5, F1, balanced acc, 分层)")
    print(f"  - cv_noise_per_fold.csv    5 折噪声综合指标")
    print(f"  - cv_summary.csv           汇总 (mean +/- std)")
    print(f"  - inference_metrics.csv    推理速度")
    print(f"  - fold{{N}}/                Dense 头模型 + 预测")
    print(f"[契约导出] {cfg.EMB_DIR} (给 LightGBM 队友)")
    print(f"  - embeddings.npz           全量嵌入 (仅原始样本, 不含增强, D={cfg.EMBED_DIM})")
    print(f"  - embeddings_clean.npz     测试集 clean (filenames + emb)")
    print(f"  - embeddings_5db.npz       测试集 5dB 噪声")
    print(f"  - embeddings_0db.npz       测试集 0dB 噪声")
    print(f"  - embeddings_m5db.npz      测试集 -5dB 噪声")
    print(f"  - label_map.json           标签映射")
    print(f"  - fold{{N}}_train/val.npz   每折切片 (train 含折内增强+类权重, 供 LightGBM)")


main()